# 9.1 Specifying Data for Simple Entities

This section covers the most fundamental ways to supply data to AMPL sets and scalar parameters using amplpy. Whether the data is a short list typed directly in Python or a value derived from an earlier computation, the interface follows a consistent and simple pattern: AMPL entities are accessed by name through the `ampl` object, and values are assigned using Python's standard assignment operator or a small number of dedicated methods.

We begin with sets, since their contents establish the index structure on which all other data depends. Once a set is populated, parameters indexed over it can be assigned their values using the same basic mechanisms. The running example throughout this section is the diet model from [Chapter 2](../02/02.md), in which the set `FOOD` names all food items available in the optimization, and the parameter `cost` records the unit price of each.

In [2]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Note: you may need to restart the kernel to use updated packages.
Licensed to AMPL Community Edition License for <uykun96@gmail.com>.


## 9.1.1 Assigning a Python List to an AMPL Set

After declaring the set `FOOD` in amplpy,

In [3]:
ampl.eval("set FOOD;")

you can assign values to it directly from Python using a standard list. The `ampl.set` attribute provides dictionary-like access to all declared AMPL sets; assigning to `ampl.set["FOOD"]` transfers the list elements into the AMPL set of that name.

For example, the following list specifies the eight food items in the diet problem:

In [4]:
ampl.set["FOOD"] = ['BEEF', 'CHK', 'FISH', 'HAM', 'MCH', 'MTL', 'SPG', 'TUR']  # Set FOOD

Each element of the list becomes a member of the AMPL set `FOOD`. The order in which elements appear in the list is not significant for the purposes of set membership, since sets in AMPL are inherently unordered collections. Any duplicates in the list are silently ignored.

This approach is particularly useful for small, hand-crafted datasets or when the set elements are generated programmatically—for instance, by reading column names from a file or by computing a range of values. More generally, any Python iterable—a list, tuple, or Python set—can be used in the same way, making it easy to populate AMPL sets from whatever data structure the surrounding code naturally produces.

## 9.1.2 Deriving a Set from Dictionary Keys

In many practical situations, the elements of a set are already implicitly present as the keys of a Python dictionary. This is common when the dictionary stores parameter values indexed by the set elements—the cost per food item being a typical example. Deriving the set directly from the dictionary keys avoids maintaining a separate list and ensures that the set and the parameter data are always consistent with each other.

Suppose the unit cost of each food item is stored in the following dictionary:

In [5]:
# Define the cost parameter for each food item
cost = {
"BEEF": 3.59,  # Cost per unit of BEEF
"CHK":  2.59,  # -//- CHK
"FISH": 2.29,  # -//- FISH
"HAM":  2.89,  # -//- HAM
"MCH":  1.89,  # -//- MCH
"MTL":  1.99,  # -//- MTL
"SPG":  1.99,  # -//- SPG
"TUR":  2.49}  # -//- TUR

Since each key in this dictionary is a food item, the keys can be passed directly to populate the AMPL set `FOOD`. Python's built-in `keys()` method returns a view of the dictionary's keys, which is then converted to a list and assigned to the set:

In [6]:
ampl.set["FOOD"] = list(cost.keys()) # Set the 'FOOD' set in AMPL using the keys from the foods dictionary

This approach is both concise and robust. Because the set membership is derived directly from the dictionary, any subsequent addition or removal of food items in the dictionary is automatically reflected in the AMPL set when the assignment is re-executed. There is no risk of the set and the parameter data falling out of step. The pattern generalises to any dictionary whose keys name the members of an AMPL set: simply convert `dict.keys()` to a list and assign it.

## 9.1.3 Using the Set Object's `set_values()` Method

The two approaches above assign values through the `ampl.set` dictionary, which offers a concise single-line syntax. An alternative is to work directly with the *Set object* that amplpy exposes for each declared AMPL set. Accessing the set this way returns a Python object with its own methods, of which `set_values()` explicitly loads a collection of values into the set.

After declaring the set `A` in AMPL,

In [7]:
ampl.eval("set A;")

you can retrieve the corresponding Set object through `ampl.set['A']` and then call `set_values()` on it to assign its elements:

The argument passed to `set_values()` can be any iterable. In the example below, a Python list `[1, 2, 3]` is used, so the integers 1, 2, and 3 become members of the AMPL set `A`:

In [8]:
A = ampl.set['A']
A.set_values([1, 2, 3])

The two-step pattern—first obtain the Set object, then call `set_values()`—is more verbose than direct assignment, but it is useful when you want finer programmatic control. For example, you might obtain the Set object once, store it in a variable, and call `set_values()` repeatedly at different points in a workflow as the data evolves. The object-oriented interface also makes it easier to chain further operations, such as reading the current set contents back into Python using the Set object's corresponding getter methods.

## 9.1.4 Assigning a Value to a Scalar Parameter

A *scalar parameter* is a single numerical value with no index. Such parameters are common in optimization models as global constants—a minimum batch size, a tax rate, a planning horizon. In AMPL, a scalar parameter is declared without an indexing expression:

```ampl
param percent;
```

After this declaration, `percent` names a single real value that can be referred to anywhere in the model. To assign a value from Python, there are two equivalent approaches.

In [9]:
ampl.eval("param percent;")

The first approach uses the `ampl.param` dictionary, which provides direct access to any declared parameter by name. Assigning a scalar to `ampl.param["percent"]` immediately sets the corresponding AMPL parameter:

In [ ]:
ampl.param["percent"] = 0.42

# Alternatively, retrieve the Parameter object and call set():
p = ampl.get_parameter("percent")
p.set(0.42)

Both lines set the AMPL parameter `percent` to the value 0.42. The first form, `ampl.param["percent"] = 0.42`, is the most concise and is generally preferred for one-off assignments. The second form—retrieving a Parameter object via `ampl.get_parameter()` and calling its `set()` method—is more explicit and mirrors the object-oriented style of Section 9.1.3. It is particularly useful when you want to store a reference to the parameter and update it repeatedly, for example in a loop that solves the model under different parameter values.

It is important to note that `ampl.get_parameter("percent")` returns an amplpy *Parameter object*, not a plain Python number. Assigning a new value to the Python variable that holds this object does **not** update the AMPL parameter; only `set()` or `ampl.param[...] = ...` communicates the value back to AMPL.